In [1]:
!pip install -q transformers accelerate bitsandbytes gradio

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import gradio as gr

# 1. Auth
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(hf_token)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00:00:0100:01


In [2]:
# 2. Quantized Config for T4 GPU
model_id = "google/medgemma-1.5-4b-it"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 3. Load Model
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

In [3]:
# 4. API Logic
def api_predict(text, image=None):
    if image:
        prompt = f"<image>\nUser: {text}\nAssistant:"
        inputs = processor(text=prompt, images=image, return_tensors="pt").to("cuda")
    else:
        prompt = f"User: {text}\nAssistant:"
        inputs = processor(text=prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=512)
    
    response = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response.split("Assistant:")[-1].strip()

In [4]:
# 5. Launch with API enabled
demo = gr.Interface(
    fn=api_predict,
    inputs=[gr.Textbox(label="Text"), gr.Image(type="pil", label="Image")],
    outputs=gr.Textbox(label="Response")
)

In [5]:
# This generates the .gradio.live link you need for Expo
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://0f9900aa8f37a62d2e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
